In [16]:
import pandas as pd
import numpy as np
from scipy import sparse

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef,
    roc_auc_score, average_precision_score,
)

pool = pd.read_parquet("data/pool_splits.parquet")
comp = pd.read_parquet("data/features_comp.parquet")
X_kmer = sparse.load_npz("data/features_kmer.npz")

y = pool["label"].to_numpy()
Xc = comp.to_numpy()

print(pool.shape, Xc.shape, X_kmer.shape)

(69420, 16) (69420, 20) (69420, 8000)


In [17]:
def score(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }

def full_score(y_true, pred, prob):
    s = score(y_true, pred)
    s["roc_auc"] = roc_auc_score(y_true, prob)
    s["pr_auc"] = average_precision_score(y_true, prob)
    return s

In [18]:
def masks(split_col):
    is_test = (pool[split_col] == "test").to_numpy()
    return ~is_test, is_test

tr_rand, te_rand = masks("split_random")
tr_clu, te_clu = masks("split_cluster")

print(tr_rand.sum(), te_rand.sum(), tr_clu.sum(), te_clu.sum())

55535 13885 55535 13885


In [19]:
def run_comp(train_mask, test_mask):
    scaler = StandardScaler().fit(Xc[train_mask])
    Xtr = scaler.transform(Xc[train_mask])
    Xte = scaler.transform(Xc[test_mask])
    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr, y[train_mask])
    return clf, clf.predict(Xte), clf.predict_proba(Xte)[:, 1]

def run_kmer(train_mask, test_mask, C=100):
    clf = LogisticRegression(max_iter=5000, solver="liblinear", C=C)
    clf.fit(X_kmer[train_mask], y[train_mask])
    return clf, clf.predict(X_kmer[test_mask]), clf.predict_proba(X_kmer[test_mask])[:, 1]

In [20]:
results = {}
models = {}
preds = {}

for name, (tr, te) in [("random", (tr_rand, te_rand)),
                       ("clustered", (tr_clu, te_clu))]:
    clf, p, pr = run_comp(tr, te)
    results[(name, "comp")] = full_score(y[te], p, pr)
    models[(name, "comp")] = clf
    preds[(name, "comp")] = (p, pr)
    print(name, "comp done")

    clf, p, pr = run_kmer(tr, te)
    results[(name, "kmer")] = full_score(y[te], p, pr)
    models[(name, "kmer")] = clf
    preds[(name, "kmer")] = (p, pr)
    print(name, "kmer done")

table = pd.DataFrame(results).T
table

random comp done
random kmer done
clustered comp done
clustered kmer done


accuracy  balanced_acc        f1       mcc   roc_auc    pr_auc
random    comp  0.610947      0.560686  0.355062  0.153854  0.620782  0.524647
          kmer  0.640043      0.601100  0.458739  0.230988  0.665668  0.590543
clustered comp  0.604681      0.555138  0.350337  0.138299  0.615874  0.521106
          kmer  0.642636      0.604108  0.464263  0.237217  0.666821  0.578276

In [21]:
table.to_csv("data/results_logistic.csv")

np.savez(
    "data/preds_logistic.npz",
    **{f"{k[0]}_{k[1]}_pred": v[0] for k, v in preds.items()},
    **{f"{k[0]}_{k[1]}_prob": v[1] for k, v in preds.items()},
)

In [10]:
import warnings
warnings.simplefilter("always")

clf = LogisticRegression(max_iter=1000, solver="liblinear")
clf.fit(X_kmer[tr_rand], y[tr_rand])
print(clf.n_iter_)

[6]


In [11]:
p, pr = preds[("random", "kmer")]
print(pr.min(), pr.max(), pr.mean())
print(p.sum())

0.3707437521380676 0.5319329642511798 0.4172288561699948
2


In [12]:
def run_kmer(train_mask, test_mask, C=100):
    clf = LogisticRegression(max_iter=5000, solver="liblinear", C=C)
    clf.fit(X_kmer[train_mask], y[train_mask])
    return clf, clf.predict(X_kmer[test_mask]), clf.predict_proba(X_kmer[test_mask])[:, 1]

In [13]:
for C in [1, 10, 100, 1000]:
    clf = LogisticRegression(max_iter=5000, solver="liblinear", C=C)
    clf.fit(X_kmer[tr_rand], y[tr_rand])
    prob = clf.predict_proba(X_kmer[te_rand])[:, 1]
    pred = clf.predict(X_kmer[te_rand])
    print(C, clf.n_iter_, round(roc_auc_score(y[te_rand], prob), 4),
          round(f1_score(y[te_rand], pred), 4), pred.sum())

1 [6] 0.6497 0.0007 2
10 [9] 0.6624 0.2066 1009
100 [12] 0.6657 0.4587 3439
1000 [15] 0.6489 0.4999 4515


In [14]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, StratifiedGroupKFold

def tune_kmer(train_mask, groups=None):
    if groups is None:
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        fit_params = {}
    else:
        cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
        fit_params = {"groups": groups[train_mask]}

    gs = GridSearchCV(
        LogisticRegression(max_iter=5000, solver="liblinear"),
        {"C": [1, 10, 100]},
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
    )
    gs.fit(X_kmer[train_mask], y[train_mask], **fit_params)
    return gs.best_params_["C"], gs.best_score_

In [15]:
clusters = pool["cluster"].to_numpy()

C_rand, s_rand = tune_kmer(tr_rand)
C_clu, s_clu = tune_kmer(tr_clu, groups=clusters)
print(C_rand, s_rand)
print(C_clu, s_clu)

100 0.6582219187163556
100 0.6537839486031457


In [22]:
def tune_comp(train_mask, groups=None):
    if groups is None:
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        fit_params = {}
    else:
        cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
        fit_params = {"groups": groups[train_mask]}

    scaler = StandardScaler().fit(Xc[train_mask])
    Xtr = scaler.transform(Xc[train_mask])

    gs = GridSearchCV(
        LogisticRegression(max_iter=2000),
        {"C": [0.01, 0.1, 1, 10]},
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
    )
    gs.fit(Xtr, y[train_mask], **fit_params)
    return gs.best_params_["C"], gs.best_score_

print(tune_comp(tr_rand))
print(tune_comp(tr_clu, groups=clusters))

(10, np.float64(0.618816203637034))
(1, np.float64(0.6183585895886134))
